## Consuming Secrets & built-in types

Consuming a Secret is **identical to a ConfigMap** — three ways:

```yaml
# 1. one env var
env:
  - name: DB_PASSWORD
    valueFrom:
      secretKeyRef: { name: db-credentials, key: password }
# 2. bulk env
envFrom:
  - secretRef: { name: db-credentials }
# 3. files in a volume
volumes:
  - name: secret-vol
    secret:
      secretName: db-credentials
      defaultMode: 0400       # owner-read only, for sensitive files
```

Two Secret-specific volume knobs: **`defaultMode`** (prefer `0400` over the `0644` default) and **`tmpfs` backing** (automatic — RAM, not disk). Update propagation is the same: env vars baked at start, volumes live-refresh.

> **Don't `echo` Secrets into logs.** The most common leak is `echo $DB_PASSWORD` in an entrypoint, or a framework logging every env var at boot. Pod stdout flows to node disk and your logging pipeline.

### Built-in types

A Secret's `type` hints at *what's inside*; some controllers check it. The ones you'll meet:

| Type | Contents | Used by |
|---|---|---|
| `Opaque` *(default)* | arbitrary key/value | most user secrets |
| `kubernetes.io/service-account-token` | a JWT | auto per ServiceAccount |
| `kubernetes.io/dockerconfigjson` | a `~/.docker/config.json` | image pull |
| `kubernetes.io/tls` | `tls.crt` + `tls.key` | Ingress, HTTPS servers |
| `kubernetes.io/basic-auth`, `ssh-auth` | by convention | tooling |

The two you'll create most are **TLS** (`kubectl create secret tls`, consumed by Ingress in notebook 09) and **dockerconfigjson** (image pull — next section). On our map this is **Secret** projecting into the **env / mount** chip, exactly like ConfigMap did.